In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import missingno as msno
import warnings; warnings.filterwarnings("ignore")
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option('display.float_format', '{:.4f}'.format)
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root))
path = "D:/Global_Superstore_Project/data/clean_orders.csv"

In [2]:
df = pd.read_csv(path)

In [3]:
df_new = df.copy()

In [4]:
df_new['order_date'] = pd.to_datetime(df_new['order_date'])
df_new['ship_date'] = pd.to_datetime(df_new['ship_date'])

In [5]:
df_new['shipping_days'] = (df_new['ship_date'] - df_new['order_date']).dt.days

In [6]:
df_new['order_year'] = df_new['order_date'].dt.year

In [7]:
df_new['order_month'] = df_new['order_date'].dt.month

In [8]:
df_new['quarter'] = df_new['order_date'].dt.quarter

In [9]:
df_new['weekday'] = df_new['order_date'].dt.day_name()

In [10]:
df_new['profit_margin'] = df_new['profit'] / df_new['sales']

In [11]:
df_new['is_discounted'] = df_new['discount'] > 0

In [12]:
df_new['sales_per_unit'] = (df_new['sales'] / df_new['quantity'])

In [14]:
df_new['discount_category'] = pd.cut(
    df_new['discount'],
    bins=[-0.01, 0, 0.2, 0.5, 1],
    labels=['No Discount', 'Low', 'Medium', 'High']
)

In [15]:
df_new['loss_order'] = np.where(df_new['profit'] < 0, 'Yes', 'No')

In [16]:
df_new

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,city,state,country,postal_code,market,region,product_id,category,sub-category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,shipping_days,order_year,order_month,quarter,weekday,profit_margin,is_discounted,sales_per_unit,discount_category,loss_order
0,32298,CA-2012-124891,2012-07-31,2012-07-31,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,United States,10024.0000,US,East,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.6500,7,0.0000,762.1845,933.5700,Critical,0,2012,7,3,Tuesday,0.3300,False,329.9500,No Discount,No
1,26341,IN-2013-77878,2013-02-05,2013-02-07,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,Australia,NaN,APAC,Oceania,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.3950,9,0.1000,-288.7650,923.6300,Critical,2,2013,2,1,Tuesday,-0.0778,True,412.1550,Low,Yes
2,25330,IN-2013-71249,2013-10-17,2013-10-18,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,Australia,NaN,APAC,Oceania,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.1710,9,0.1000,919.9710,915.4900,Medium,1,2013,10,4,Thursday,0.1778,True,575.0190,Low,No
3,13524,ES-2013-1579342,2013-01-28,2013-01-30,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,Germany,NaN,EU,Central,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.5100,5,0.1000,-96.5400,910.1600,Medium,2,2013,1,1,Monday,-0.0334,True,578.5020,Low,Yes
4,47221,SG-2013-4320,2013-11-05,2013-11-06,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,Senegal,NaN,Africa,Africa,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.9600,8,0.0000,311.5200,903.0400,Critical,1,2013,11,4,Tuesday,0.1100,False,354.1200,No Discount,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51285,29002,IN-2014-62366,2014-06-19,2014-06-19,Same Day,KE-16420,Katrina Edelman,Corporate,Kure,Hiroshima,Japan,NaN,APAC,North Asia,OFF-FA-10000746,Office Supplies,Fasteners,"Advantus Thumb Tacks, 12 Pack",65.1000,5,0.0000,4.5000,0.0100,Medium,0,2014,6,2,Thursday,0.0691,False,13.0200,No Discount,No
51286,35398,US-2014-102288,2014-06-20,2014-06-24,Standard Class,ZC-21910,Zuschuss Carroll,Consumer,Houston,Texas,United States,77095.0000,US,Central,OFF-AP-10002906,Office Supplies,Appliances,Hoover Replacement Belt for Commercial Guardsm...,0.4440,1,0.8000,-1.1100,0.0100,Medium,4,2014,6,2,Friday,-2.5000,True,0.4440,High,Yes
51287,40470,US-2013-155768,2013-12-02,2013-12-02,Same Day,LB-16795,Laurel Beltran,Home Office,Oxnard,California,United States,93030.0000,US,West,OFF-EN-10001219,Office Supplies,Envelopes,"#10- 4 1/8"" x 9 1/2"" Security-Tint Envelopes",22.9200,3,0.0000,11.2308,0.0100,High,0,2013,12,4,Monday,0.4900,False,7.6400,No Discount,No
51288,9596,MX-2012-140767,2012-02-18,2012-02-22,Standard Class,RB-19795,Ross Baird,Home Office,Valinhos,São Paulo,Brazil,NaN,LATAM,South,OFF-BI-10000806,Office Supplies,Binders,"Acco Index Tab, Economy",13.4400,2,0.0000,2.4000,0.0030,Medium,4,2012,2,1,Saturday,0.1786,False,6.7200,No Discount,No


In [ ]:
df_new.to_csv("D:/Global_Superstore_Project/data/processed/featured_orders.csv", index=False)